[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/01_python_for_ml/04_pandas_essentials.ipynb)

# Bài 04 — Pandas cho Machine Learning

**🎯 Mục tiêu**
- Đọc, khám phá, lọc, biến đổi dữ liệu bảng bằng `DataFrame`.
- Thành thạo `loc`/`iloc`, boolean mask, `groupby`, `merge`, `pivot_table`.
- Xử lý missing value cơ bản và chuyển DataFrame sang NumPy để đưa vào model.
- Tránh các bẫy hay gặp: `SettingWithCopyWarning`, vòng lặp `iterrows`, sai kiểu dữ liệu.

Dữ liệu dùng trong bài: `data/sample/customer_churn.csv` — khách hàng của một công ty viễn thông, cột `churn = 1` nghĩa là khách hàng đã rời bỏ dịch vụ.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/01_python_for_ml

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
print(pd.__version__)

## 1. Series và DataFrame

- **Series**: mảng 1 chiều có **nhãn** (index).
- **DataFrame**: bảng 2 chiều — tập hợp các Series cùng index, mỗi cột có kiểu riêng.

In [ ]:
s = pd.Series([0.81, 0.77, 0.85], index=["knn", "tree", "logreg"], name="accuracy")
print(s)
print("Tốt nhất:", s.idxmax(), s.max())

df_small = pd.DataFrame({
    "model": ["knn", "tree", "logreg"],
    "accuracy": [0.81, 0.77, 0.85],
    "train_time_s": [0.01, 0.2, 0.5],
})
df_small

## 2. Đọc dữ liệu và khám phá nhanh

Thói quen đầu tiên với mọi dataset: `shape`, `head`, `info`, `describe`, `value_counts`.

In [ ]:
df = pd.read_csv("../../data/sample/customer_churn.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print(df["contract"].value_counts())
print()
print(df["churn"].value_counts(normalize=True).round(3))   # tỉ lệ churn

## 3. Chọn dữ liệu: cột, hàng, `loc`, `iloc`

| Cú pháp | Ý nghĩa |
|---|---|
| `df["col"]` | một cột → Series |
| `df[["a", "b"]]` | nhiều cột → DataFrame |
| `df.loc[nhãn_hàng, nhãn_cột]` | chọn theo **nhãn** (slice *gồm* điểm cuối) |
| `df.iloc[vị_trí_hàng, vị_trí_cột]` | chọn theo **vị trí** số nguyên (giống NumPy) |
| `df[mask]` | lọc hàng bằng boolean mask |

In [ ]:
print(df["monthly_charges"].head(3))
df[["tenure_months", "contract", "churn"]].head(3)

In [ ]:
print(df.loc[0:2, ["customer_id", "contract"]])   # nhãn 0..2 (gồm 2)
print(df.iloc[0:2, 0:3])                          # vị trí 0..1 (không gồm 2)

**Lọc bằng điều kiện** — dùng `&`, `|`, `~` và đặt mỗi điều kiện trong ngoặc (giống NumPy). `query()` là cách viết dễ đọc hơn:

In [ ]:
risky = df[(df["contract"] == "month-to-month") & (df["support_calls"] >= 4)]
print(len(risky), "khách hàng rủi ro; tỉ lệ churn:", round(risky["churn"].mean(), 3))

same = df.query("contract == 'month-to-month' and support_calls >= 4")
print(len(same) == len(risky))

print(df[df["payment_method"].isin(["e-wallet", "cash"])].shape)

## 4. Tạo và biến đổi cột

Luôn ưu tiên phép toán **vectorized** trên cả cột (giống NumPy) thay vì vòng lặp.

In [ ]:
df = df.copy()
df["avg_charge_per_month"] = df["total_charges"] / df["tenure_months"].clip(lower=1)
df["is_long_term"] = df["contract"] != "month-to-month"
df["tenure_group"] = pd.cut(df["tenure_months"], bins=[-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"])
df["internet_service"] = df["internet_service"].map({"dsl": "DSL", "fiber": "Fiber", "none": "None"})
df[["tenure_months", "tenure_group", "avg_charge_per_month", "is_long_term", "internet_service"]].head()

⚠️ **`SettingWithCopyWarning`**: khi gán vào một DataFrame được *lọc* ra từ DataFrame khác, pandas không chắc bạn đang sửa bản gốc hay bản sao. Hai quy tắc an toàn:
1. Gán bằng `df.loc[mask, "col"] = giá_trị` trên DataFrame gốc;
2. Khi muốn làm việc với tập con, tạo bản sao rõ ràng: `sub = df[mask].copy()`.

In [ ]:
df.loc[df["support_calls"] > 6, "support_calls"] = 6     # cắt ngưỡng (capping) — cách đúng
sub = df[df["senior"]].copy()
sub["note"] = "senior"                                      # an toàn vì sub là bản sao
print(df["support_calls"].max(), sub.shape)

⚠️ **Đừng dùng `iterrows()` để tính toán** — chậm hơn phép toán vectorized hàng trăm lần:

In [ ]:
import time

start = time.perf_counter()
slow = [row["monthly_charges"] * 12 for _, row in df.iterrows()]
t_slow = time.perf_counter() - start

start = time.perf_counter()
fast = df["monthly_charges"] * 12
t_fast = time.perf_counter() - start
print(f"iterrows: {t_slow * 1000:.1f} ms | vectorized: {t_fast * 1000:.2f} ms | nhanh hơn ~{t_slow / t_fast:.0f} lần")

## 5. `groupby` — "split → apply → combine"

Câu hỏi EDA kinh điển: **tỉ lệ churn khác nhau thế nào giữa các nhóm khách hàng?**

In [ ]:
df.groupby("contract")["churn"].mean().sort_values(ascending=False).round(3)

In [ ]:
summary = df.groupby(["contract", "internet_service"]).agg(
    n_customers=("customer_id", "count"),
    churn_rate=("churn", "mean"),
    avg_monthly=("monthly_charges", "mean"),
).round(3)
summary

In [ ]:
pd.pivot_table(df, values="churn", index="tenure_group", columns="contract", aggfunc="mean", observed=True).round(3)

`transform` trả về kết quả **cùng độ dài** với DataFrame gốc — hữu ích để tạo feature "so với nhóm":

In [ ]:
df["charge_vs_contract_mean"] = df["monthly_charges"] - df.groupby("contract")["monthly_charges"].transform("mean")
df[["contract", "monthly_charges", "charge_vs_contract_mean"]].head()

## 6. Kết hợp bảng: `merge` và `concat`

Dữ liệu thực tế nằm ở nhiều bảng. `merge` giống `JOIN` trong SQL:

In [ ]:
plans = pd.DataFrame({
    "contract": ["month-to-month", "one_year", "two_year"],
    "min_months": [1, 12, 24],
    "discount": [0.0, 0.05, 0.12],
})
merged = df.merge(plans, on="contract", how="left", validate="many_to_one")
print(merged.shape)
merged[["customer_id", "contract", "min_months", "discount"]].head(3)

> 💡 `validate="many_to_one"` bắt lỗi khi bảng bên phải có key trùng lặp — nếu không, `merge` sẽ âm thầm **nhân bản** hàng, một lỗi dữ liệu rất khó phát hiện.

| `how=` | Giữ lại |
|---|---|
| `inner` | chỉ key có ở cả hai bảng |
| `left` | mọi hàng của bảng trái |
| `right` | mọi hàng của bảng phải |
| `outer` | mọi key ở cả hai bảng |

In [ ]:
part1, part2 = df.iloc[:3], df.iloc[3:6]
print(pd.concat([part1, part2], axis=0).shape)   # nối theo hàng

## 7. Missing values

In [ ]:
houses = pd.read_csv("../../data/sample/house_prices_dirty.csv")
print(houses.isna().sum())
print("\nTỉ lệ missing:\n", houses.isna().mean().round(3))

In [ ]:
filled = houses.copy()
filled["age_years"] = filled["age_years"].fillna(filled["age_years"].median())
filled = filled.dropna(subset=["bathrooms"])           # bỏ hàng thiếu bathrooms
print(filled.isna().sum().sum(), "giá trị còn thiếu;", len(houses) - len(filled), "hàng bị bỏ")

⚠️ Điền missing value bằng median của **toàn bộ** dữ liệu trước khi chia train/test là một dạng **data leakage**. Cách làm đúng (chỉ học median từ train) sẽ được học ở chương 04 và 05.

Dữ liệu bẩn còn có category viết không thống nhất:

In [ ]:
print(houses["district"].value_counts())
houses["district_clean"] = (houses["district"].str.strip().str.replace("Q.", "Quận", regex=False).str.title())
print(houses["district_clean"].value_counts())

## 8. Ngày tháng

In [ ]:
sales = pd.read_csv("../../data/sample/sales_daily.csv", parse_dates=["date"])
sales["dayofweek"] = sales["date"].dt.day_name()
sales["month"] = sales["date"].dt.month
print(sales.groupby("dayofweek")["sales"].mean().sort_values().round(1))
monthly = sales.set_index("date")["sales"].resample("MS").sum()
monthly.head()

**Vẽ nhanh bằng pandas.** `Series`/`DataFrame` có sẵn `.plot()` (gọi Matplotlib bên dưới — bài 05 học kỹ): tiện để nhìn ngay kết quả `groupby` hay một chuỗi thời gian.

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
df.groupby("contract")["churn"].mean().sort_values().plot(kind="barh", ax=axes[0], color="C3")
axes[0].set_xlabel("tỉ lệ churn")
axes[0].set_title("Tỉ lệ churn theo loại hợp đồng (groupby)")
monthly.plot(ax=axes[1], marker="o", ms=3)
axes[1].set_ylabel("tổng doanh số tháng")
axes[1].set_title("Doanh số theo tháng (resample)")
plt.tight_layout()
plt.show()

## 9. Từ DataFrame sang NumPy

Model nhận ma trận số. Chọn feature → mã hoá category → `to_numpy()`:

In [ ]:
features = ["tenure_months", "monthly_charges", "support_calls", "n_products"]
X = df[features].to_numpy(dtype=float)
y = df["churn"].to_numpy()
X_cat = pd.get_dummies(df[["contract", "internet_service"]], dtype=float)   # one-hot (chương 04 học kỹ)
X_full = np.hstack([X, X_cat.to_numpy()])
print(X.shape, X_cat.shape, X_full.shape, y.shape)
print(list(X_cat.columns))

## 🧠 Kiểm tra nhanh

1. `df.loc[0:2]` và `df.iloc[0:2]` trả về bao nhiêu hàng? Vì sao khác nhau?
2. Viết code tính tỉ lệ churn theo `payment_method`, sắp xếp giảm dần.
3. Khi nào `merge` làm **tăng** số hàng ngoài ý muốn? Phòng tránh thế nào?
4. Vì sao không nên điền missing bằng median của toàn bộ dataset trước khi chia train/test?

## 📌 Tóm tắt

- Khám phá: `shape`, `head`, `info`, `describe`, `value_counts`, `isna().mean()`.
- Chọn: `loc` (nhãn), `iloc` (vị trí), boolean mask / `query`.
- Biến đổi: phép toán vectorized, `map`, `cut`, `str.*`, `dt.*`; tránh `iterrows`.
- Tổng hợp: `groupby().agg()`, `transform`, `pivot_table`; kết hợp: `merge(validate=...)`, `concat`.
- Cuối cùng: `to_numpy()` / `get_dummies` để đưa vào model.

## 📚 Tài liệu tham khảo

- 10 minutes to pandas — https://pandas.pydata.org/docs/user_guide/10min.html
- Wes McKinney, *Python for Data Analysis*, 3rd ed. (miễn phí online: https://wesmckinney.com/book/)